# Can we predict `passed_quiz`?
Follows `protocol.yaml`: group holdout on `student_key` (`holdout.csv`), 5-fold GroupKFold CV inside training rows, selection by mean CV ROC AUC, holdout scored once.

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
df=pd.read_csv('data.csv'); hold=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==12453 and len(hold)==2465
train_idx=np.setdiff1d(np.arange(len(df)),hold)
assert not set(df.student_key[hold])&set(df.student_key[train_idx])
tr=df.iloc[train_idx]; ho=df.iloc[hold]
print(len(tr),len(ho),tr.passed_quiz.mean(),ho.passed_quiz.mean())

## Features and prediction time
`student_key` is excluded (entity key). `session_minutes` is the length of the *whole* session, which is closed by the quiz, so it is only final after the outcome is known: **excluded** as not known at prediction time. `hints_used`, `prior_score`, `enrol_year`, `school_code` are available before the quiz result and are kept. (`hints_used` is the one judgement call: if hints can also be opened during the quiz it would need to go too.)

In [ ]:
FEATURES=['enrol_year','school_code','prior_score','hints_used']
TARGET='passed_quiz'
X=tr[FEATURES]; y=tr[TARGET].to_numpy(); g=tr.student_key.to_numpy()

## Candidates (named before any scoring)
All preprocessing sits inside pipelines, so it is fitted within each CV fold.

In [ ]:
candidates={
 'baseline_prior':lambda: DummyClassifier(strategy='prior'),
 'logreg_C0.1':lambda: make_pipeline(StandardScaler(),LogisticRegression(C=0.1,max_iter=1000)),
 'logreg_C1':lambda: make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=1000)),
 'hgb_shallow':lambda: HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,random_state=0),
 'hgb_deeper':lambda: HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=150,min_samples_leaf=40,random_state=0),
}
print(list(candidates))

In [ ]:
folds=list(GroupKFold(n_splits=5).split(X,y,g))
cv_scores={}
for name,mk in candidates.items():
    s=[]
    for a,b in folds:
        m=mk().fit(X.iloc[a],y[a]); s.append(roc_auc_score(y[b],m.predict_proba(X.iloc[b])[:,1]))
    cv_scores[name]=s
    print(f'{name:16s} mean={np.mean(s):.4f} sd={np.std(s):.4f}')
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k])); print('chosen:',chosen)

## Refit on all training rows, score once on the holdout

In [ ]:
final=candidates[chosen]().fit(X,y)
holdout_auc=roc_auc_score(ho[TARGET],final.predict_proba(ho[FEATURES])[:,1])
print(f'holdout roc_auc = {holdout_auc:.4f}')
evidence={'features':FEATURES,'candidates':list(candidates),
 'cv_folds':[[train_idx[a].tolist(),train_idx[b].tolist()] for a,b in folds],
 'cv_scores':cv_scores,'chosen':chosen,'holdout_rows':hold.tolist(),
 'holdout_scores':{chosen:holdout_auc}}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the printed CV and holdout AUC above; compare with the 0.5 baseline to judge whether the quiz outcome is predictable from pre-quiz information.